# 1. Title

**Prepared for:** Mohammed (MM)  
**Dataset:** Titanic passenger records (Data Science Dojo)

This project builds a reusable Python workflow for loading, cleaning, and exploring Titanic passenger data. It compares survival across passenger class, recorded sex, and family status, then examines how missing-age handling affects descriptive conclusions.

**Authorship disclosure:** This is an AI-assisted implementation prepared with Codex; Mohammed must review, understand, and adapt the work before submission under the course authorship rules.

## 2. Setup
Use deterministic operations and a fixed random seed. Run all cells from this directory; no network download is required.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

np.random.seed(42)
sns.set_theme(style="whitegrid", palette="colorblind")
pd.set_option("display.max_columns", 20)
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)


## 3. Ingestion
Load the supplied CSV unchanged. Record its initial shape, missing values, duplicate count, and data types before cleaning.

In [ ]:
df = pd.read_csv("titanic.csv")
display(df.head())
display(df.shape)
print("Duplicate rows:", df.duplicated().sum())
display(df.isna().sum().rename("Missing values").to_frame())
display(df.dtypes.astype(str).rename("Data type").to_frame())
print("Numeric columns:", len(df.select_dtypes(include="number").columns))
print("Text columns:", len(df.select_dtypes(exclude="number").columns))
assert df.shape == (891, 12)
assert df["PassengerId"].is_unique
assert df["Survived"].isin([0, 1]).all()


## 4. Cleaning functions
Keep all passengers. Fill missing ages with the overall observed median (robust to extreme ages), and preserve missingness in an `AgeWasMissing` flag. Mark missing Cabin and Embarked as `Unknown`; this is a missingness label, not an inferred cabin or port. Derive family size and alone status without modifying the raw frame. Fare is not imputed because it has no missing values.

In [ ]:
def handle_missing_values(frame):
    """Copy the frame, flag and median-fill Age, and label absent Cabin/Embarked."""
    cleaned = frame.copy(deep=True)
    cleaned["AgeWasMissing"] = cleaned["Age"].isna()
    age_median = cleaned["Age"].median()
    if pd.isna(age_median):
        raise ValueError("Age requires at least one observed value for median imputation.")
    cleaned["Age"] = cleaned["Age"].fillna(age_median)
    for column in ["Cabin", "Embarked"]:
        cleaned[column] = cleaned[column].fillna("Unknown")
    return cleaned


def derive_family_features(frame):
    """Copy the frame and derive FamilySize and boolean IsAlone from SibSp/Parch."""
    cleaned = frame.copy(deep=True)
    cleaned["FamilySize"] = cleaned["SibSp"] + cleaned["Parch"] + 1
    cleaned["IsAlone"] = cleaned["FamilySize"].eq(1)
    return cleaned


cleaned_df = derive_family_features(handle_missing_values(df))
print("Observed median Age:", df["Age"].median())
display(cleaned_df.isna().sum().rename("Missing after cleaning").to_frame())
display(cleaned_df[["Age", "AgeWasMissing", "Cabin", "Embarked", "FamilySize", "IsAlone"]].head())
assert len(cleaned_df) == len(df)
assert cleaned_df.isna().sum().sum() == 0
assert cleaned_df["AgeWasMissing"].sum() == 177
assert cleaned_df["IsAlone"].equals(cleaned_df["FamilySize"].eq(1))
assert df["Age"].isna().sum() == 177  # raw input stays unchanged


## 5. Exploratory analysis function
Report group sizes alongside survival percentages. Associations are descriptive and do not establish causal effects. Compare three age treatments using fixed under-18 and 18-plus groups; class and sex survival rates remain unchanged when only Age is filled and all rows are retained.

In [ ]:
def survival_by_group(frame, column):
    """Return passenger counts, survivor counts and survival percent by a column."""
    grouped = frame.groupby(column, dropna=False, observed=True)["Survived"].agg(
        Passengers="size", Survivors="sum", SurvivalRate="mean"
    )
    grouped["SurvivalPercent"] = 100 * grouped["SurvivalRate"]
    return grouped


display(cleaned_df.describe())
for column in ["Pclass", "Sex", "IsAlone"]:
    print("Survival grouped by", column)
    display(survival_by_group(cleaned_df, column).round(3))
print("Overall survival percent:", round(100 * cleaned_df["Survived"].mean(), 3))
print("Age missingness by passenger class (%)")
display((100 * df.groupby("Pclass")["Age"].apply(lambda ages: ages.isna().mean())).to_frame("AgeMissingPercent"))

group_age = df["Age"].fillna(df.groupby(["Pclass", "Sex"])["Age"].transform("median"))
treatments = {
    "Observed ages only": df.loc[df["Age"].notna()].copy(),
    "Overall median": cleaned_df.copy(),
    "Class/sex median": df.assign(Age=group_age),
}
sensitivity_tables = []
for method, frame in treatments.items():
    frame = frame.assign(AgeGroup=np.where(frame["Age"] < 18, "Under 18", "18 and above"))
    table = survival_by_group(frame, "AgeGroup").reset_index()
    table.insert(0, "Method", method)
    sensitivity_tables.append(table)
sensitivity = pd.concat(sensitivity_tables, ignore_index=True)
display(sensitivity.round(3))
print("Mean Age by outcome, before versus after filling")
display(pd.DataFrame({"Observed": df.groupby("Survived")["Age"].mean(),
                      "Median filled": cleaned_df.groupby("Survived")["Age"].mean()}))
